# A3.5 · Tool permission models

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** The confused deputy at the tool layer.

**Control.** Capability scoping, allowlisted actions, structured output contracts, read-only defaults.

**This lab.** Design the dangerous call out of existence.

| | |
|---|---|
| Open-source tooling | kmcp, OPA |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.5"))

Tool permission models are where L2 and L2.5 stop being vocabulary and start being configuration.

In [ ]:
from cybercommons import sandbox

MODELS = {
 "allow-all (no model at all)":
    sandbox.ToolPolicy(allow={"read_file", "write_file", "run_shell",
                              "delete_repo", "rotate_secrets"}),
 "L2 — approve every writer":
    sandbox.ToolPolicy(allow={"read_file"},
                       require_approval={"write_file", "run_shell",
                                         "delete_repo", "rotate_secrets"}),
 "L2.5 — bounded set, some tools never":
    sandbox.ToolPolicy(allow={"read_file", "write_file"},
                       require_approval={"run_shell"},
                       deny={"delete_repo", "rotate_secrets"}),
}
calls = ["read_file", "write_file", "run_shell", "delete_repo", "unknown_tool"]
for name, pol in MODELS.items():
    print(name)
    for c in calls:
        print("   ", pol.check(c))
    print()

Look at `unknown_tool` in each. Deny-by-default is the property that makes the model hold when someone adds a tool and forgets to update the policy — which is the normal case, not the exception.

### Expect

The allow-all policy permits everything named and still denies `unknown_tool`. L2 gates every writer. L2.5 allows the bounded set, gates the shell and refuses the two destructive tools outright.

### Your turn

Where does 'approve' actually happen for your agents — a human in a chat window, or a policy engine? Measure the median approval latency. If it is under two seconds, nobody is reading them.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*